In [ ]:
from datetime import datetime
import re


# Базовый класс банковского счёта
class Account:
    _account_counter = 1000

    def __init__(self, account_holder: str, balance: float = 0.0):
        # Валидация формата имени владельца
        name_pattern = (
            r"(?:[A-Z][a-z]* [A-Z][a-z]*|"
            r"[А-ЯЁ][а-яё]* [А-ЯЁ][а-яё]*)"
        )
        if not re.fullmatch(name_pattern, account_holder):
            raise ValueError(
                "Имя владельца должно быть в формате 'Имя Фамилия'"
            )

        # Валидация начального баланса
        if balance < 0:
            raise ValueError("Баланс не может быть отрицательным")

        self.holder = account_holder
        self._balance = balance

        # Формирование номера счёта
        self.account_number = f"ACC-{Account._account_counter:04d}"
        Account._account_counter += 1

        self.operations_history = []

    # Пополнение счёта
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError(
                "Сумма депозита должна быть положительной"
            )

        self._balance += amount

        operation = {
            "type": "deposit",
            "amount": amount,
            "date": datetime.now(),
            "balance": self._balance,
            "status": "success"
        }

        self.operations_history.append(operation)

    # Снятие средств со счёта
    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError(
                "Сумма снятия должна быть положительной"
            )

        if amount > self._balance:
            operation = {
                "type": "withdraw",
                "amount": amount,
                "date": datetime.now(),
                "balance": self._balance,
                "status": "fail"
            }

            self.operations_history.append(operation)
            return

        self._balance -= amount

        operation = {
            "type": "withdraw",
            "amount": amount,
            "date": datetime.now(),
            "balance": self._balance,
            "status": "success"
        }

        self.operations_history.append(operation)

    # Получение текущего баланса
    def get_balance(self):
        return self._balance

    # Получение истории операций
    def get_history(self):
        return self.operations_history.copy()

    # Вывод n крупнейших операций по дате
    def get_last_largest_operations(self, n):
        if n <= 0:
            raise ValueError(
                "Количество операций должно быть положительным"
            )

        operations = sorted(
            self.operations_history,
            key=lambda operation: (
                operation["amount"],
                operation["date"]
            ),
            reverse=True
        )[:n]

        operations.sort(
            key=lambda operation: operation["date"],
            reverse=True
        )

        for operation in operations:
            print(operation)


# Расчётный счёт
class CheckingAccount(Account):
    account_type = "checking"


# Сберегательный счёт
class SavingsAccount(Account):
    account_type = "savings"

    # Начисление процентов на остаток
    def apply_interest(self, rate):
        interest = self._balance * rate / 100
        self._balance += interest

        operation = {
            "type": "interest",
            "amount": interest,
            "date": datetime.now(),
            "balance": self._balance,
            "status": "success"
        }

        self.operations_history.append(operation)

    # Снятие не более 50% текущего баланса
    def withdraw(self, amount):
        if amount > self._balance * 0.5:
            operation = {
                "type": "withdraw",
                "amount": amount,
                "date": datetime.now(),
                "balance": self._balance,
                "status": "fail"
            }

            self.operations_history.append(operation)
            return

        super().withdraw(amount)